# Notebook 06: Self-Supervised Vision — DINO

## Why This Matters
DINO (Caron et al., 2021) produces something surprising: attention maps that segment objects without ever being trained on segmentation labels. Feed a dog photo through a DINO ViT and the attention heads find the dog's outline. Feed a street scene and it finds pedestrians, cars, sky. This emergent behavior comes purely from self-supervised pretraining on ImageNet.

DINO also produces the best linear evaluation scores of any self-supervised method at the time of release — often matching supervised ViT features. The features transfer well to retrieval, clustering, and anomaly detection, which makes DINO a practical tool for representation learning today.

## Structure
1. Self-distillation without labels — how DINO avoids collapse (narrative)
2. Loading a pretrained DINO model
3. Visualizing self-attention maps — emergent segmentation
4. Feature extraction for downstream tasks
5. Zero-shot image similarity and retrieval
6. Bridge to CLIP

## What You'll Learn
- How DINO's centering + sharpening prevents the mode collapse that plagues negative-free methods
- Why ViT patch tokens contain spatial information that CNNs don't expose the same way
- How DINO attention maps find object boundaries without segmentation supervision
- How to use DINO features for practical retrieval and clustering tasks


## Background

### Self-distillation: the DINO idea

Contrastive methods (SimCLR, MoCo) need explicit negatives to prevent collapse — without negatives, the trivially easy solution is to map everything to the same point. DINO eliminates negatives and prevents collapse differently:

**Teacher-student setup:**
- Student sees two local crops (small views of the image)
- Teacher sees two global crops (larger views covering most of the image)
- Student is trained to match the teacher's output distribution via cross-entropy

**Teacher = momentum encoder:**
- Teacher weights are an EMA of the student weights (same as MoCo)
- No gradients flow through the teacher

**Centering prevents collapse:**
- Teacher outputs are centered by subtracting a running mean: `g(x) = softmax((f(x) - c) / τ_t)`
- This ensures the teacher can't collapse to a constant — the mean subtraction would make that zero-loss-free

**Sharpening encourages confident predictions:**
- Teacher uses a low temperature τ_t ≈ 0.04 (sharp, confident distribution)
- Student uses a higher temperature τ_s ≈ 0.1 (softer, more uniform)
- The asymmetry prevents trivial solutions

### Why ViT + DINO produces segmentation

Vision Transformers process images as sequences of patches. Each patch attends to all other patches. In DINO-pretrained ViTs, the [CLS] token's attention to patch tokens concentrates on semantically meaningful regions — because the self-distillation objective implicitly rewards representations that focus on stable, informative parts of the scene across different views and crops.

This is not explicitly trained — it emerges as a consequence of the objective.


In [ ]:
# %pip install -q torch torchvision timm numpy matplotlib scikit-learn Pillow requests

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load pretrained DINO ViT-S/8 from torch.hub (Facebook Research official weights)
# ViT-S/8: small ViT with 8×8 patch size — high spatial resolution attention maps
print("Loading pretrained DINO ViT-S/8...")
dino_model = torch.hub.load("facebookresearch/dino:main", "dino_vits8", pretrained=True)
dino_model = dino_model.to(device).eval()

n_params = sum(p.numel() for p in dino_model.parameters())
print(f"DINO ViT-S/8: {n_params/1e6:.1f}M parameters")
print(f"Patch size: 8×8 pixels")
print(f"Attention heads: {dino_model.blocks[0].attn.num_heads}")

## 1. Preprocessing

In [ ]:
transform = T.Compose([
    T.Resize(480),
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_image(url_or_path: str) -> tuple:
    """Load an image and return (PIL image, tensor)."""
    if url_or_path.startswith("http"):
        resp = requests.get(url_or_path, timeout=10)
        img = Image.open(BytesIO(resp.content)).convert("RGB")
    else:
        img = Image.open(url_or_path).convert("RGB")
    return img, transform(img).unsqueeze(0)

# Use a sample image from torchvision datasets
import torchvision
dataset = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=True,
                                        transform=T.Compose([T.Resize(224), T.ToTensor(),
                                            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]))
sample_img_tensor, sample_label = dataset[0]
sample_img_tensor = sample_img_tensor.unsqueeze(0).to(device)

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"Sample image class: {classes[sample_label]}")
print(f"Tensor shape: {sample_img_tensor.shape}")

## 2. Extracting Self-Attention Maps

In [ ]:
def get_dino_attention(model, img_tensor: torch.Tensor, head_idx: int = None):
    """Extract self-attention maps from the last transformer block.
    
    Returns attention from [CLS] token to all patch tokens.
    Shape: [num_heads, h_patches, w_patches]
    """
    # Hook to capture attention weights from last block
    attentions = {}
    def hook_fn(module, input, output):
        attentions["attn"] = output

    # Register hook on last block's attention
    hook = model.blocks[-1].attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        _ = model(img_tensor)

    hook.remove()

    # Attention shape from ViT: [batch, num_heads, num_tokens, num_tokens]
    # num_tokens = 1 (CLS) + num_patches
    attn = attentions["attn"][0]  # [num_heads, num_tokens, num_tokens]
    num_heads = attn.shape[0]

    # Get attention from CLS token (index 0) to all patch tokens (index 1:)
    cls_attn = attn[:, 0, 1:]  # [num_heads, num_patches]

    # Reshape to spatial grid
    img_size = img_tensor.shape[-1]
    patch_size = 8  # ViT-S/8
    h = w = img_size // patch_size
    cls_attn = cls_attn.reshape(num_heads, h, w).cpu().numpy()

    return cls_attn


# Extract attention maps from the CIFAR sample
# Note: DINO was trained on ImageNet, not CIFAR — but attention patterns still emerge
attn_maps = get_dino_attention(dino_model, sample_img_tensor)
print(f"Attention maps shape: {attn_maps.shape}  (num_heads × h_patches × w_patches)")
print(f"Each map: {attn_maps.shape[1]}×{attn_maps.shape[2]} = {attn_maps.shape[1]*attn_maps.shape[2]} patches")

# Visualize attention maps across all heads
n_heads = attn_maps.shape[0]
fig, axes = plt.subplots(2, n_heads // 2, figsize=(14, 5))
axes = axes.flatten()

for i in range(n_heads):
    attn = attn_maps[i]
    attn = (attn - attn.min()) / (attn.max() - attn.min())
    axes[i].imshow(attn, cmap="inferno", interpolation="nearest")
    axes[i].set_title(f"Head {i+1}", fontsize=9)
    axes[i].axis("off")

plt.suptitle("DINO ViT-S/8 — CLS attention maps per head (last block)", fontsize=12)
plt.tight_layout()
plt.show()
print("Different heads specialize in different spatial regions — no supervision for this.")

## 3. Feature Extraction for Downstream Tasks

In [ ]:
def extract_dino_features(model, loader, device, use_cls=True):
    """Extract DINO features from a dataset.
    
    use_cls=True:  use [CLS] token representation (global image feature)
    use_cls=False: use mean of patch tokens (sometimes better for dense tasks)
    """
    model.eval()
    all_features, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            # get_intermediate_layers returns patch + CLS tokens from each block
            features = model.forward_features(imgs) if hasattr(model, 'forward_features')                        else model(imgs)
            # For timm-style ViT: model returns CLS token by default
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)


# Use the CIFAR-10 test set for evaluation
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

test_loader = DataLoader(
    torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=False,
        transform=T.Compose([T.Resize(224), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])),
    batch_size=64, shuffle=False, num_workers=0
)
train_loader = DataLoader(
    torchvision.datasets.CIFAR10("/tmp/cifar10", train=True, download=False,
        transform=T.Compose([T.Resize(224), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])),
    batch_size=64, shuffle=False, num_workers=0
)

print("Extracting DINO features...")
train_feats, train_labels = extract_dino_features(dino_model, train_loader, device)
test_feats,  test_labels  = extract_dino_features(dino_model, test_loader,  device)
print(f"Feature shape: {train_feats.shape}")

scaler = StandardScaler()
clf = LogisticRegression(max_iter=300, C=1.0, random_state=42)
clf.fit(scaler.fit_transform(train_feats), train_labels)
acc = accuracy_score(test_labels, clf.predict(scaler.transform(test_feats)))
print(f"\nLinear probe accuracy on CIFAR-10: {acc:.3f}")
print("(DINO trained on ImageNet — features transfer without any CIFAR fine-tuning)")

## 4. Zero-Shot Image Similarity

In [ ]:
# Use DINO features for retrieval — no fine-tuning needed
from sklearn.metrics.pairwise import cosine_similarity

# Take a small subset for visualization
n_query = 5
n_gallery = 200

gallery_feats  = test_feats[:n_gallery]
gallery_labels = test_labels[:n_gallery]
query_feats    = test_feats[:n_query]
query_labels   = test_labels[:n_query]

sims = cosine_similarity(query_feats, gallery_feats)

print("Zero-shot retrieval — top-5 matches for each query:")
print(f"{'Query class':<15} {'Top-5 retrieved classes'}")
print("-" * 60)
for i in range(n_query):
    top5_idx = np.argsort(sims[i])[::-1][1:6]  # skip self (index 0)
    top5_classes = [classes[gallery_labels[j]] for j in top5_idx]
    print(f"  {classes[query_labels[i]]:<13}: {top5_classes}")

print()
# Compute recall@1 over the full test set
sims_full = cosine_similarity(test_feats, test_feats)
np.fill_diagonal(sims_full, -1)  # exclude self
nearest = np.argmax(sims_full, axis=1)
recall_at_1 = np.mean(test_labels[nearest] == test_labels)
print(f"Recall@1 (zero-shot, CIFAR-10 test set): {recall_at_1:.3f}")
print("Without any retrieval-specific training — pure DINO features.")

## 5. Bridge to CLIP

DINO is unimodal — it learns visual representations from visual self-supervision. The teacher-student momentum pattern produces rich spatial features, but the representations have no connection to language.

**CLIP** (Radford et al., 2021) extends the contrastive framework across modalities: instead of two image views as a positive pair, CLIP uses an image and its caption. The model learns to align image and text representations in a shared embedding space.

The result: representations where you can search images with text queries, classify images with natural language descriptions, and compare content across modalities — all without task-specific fine-tuning.

Notebook 07 covers CLIP zero-shot classification and image-text retrieval.


## Summary

| Concept | Key Point |
|---------|-----------|
| Self-distillation | Student matches teacher's output distribution — no explicit negatives |
| Momentum teacher | EMA of student weights — same pattern as MoCo's key encoder |
| Centering | Subtracts running mean from teacher outputs — prevents constant-output collapse |
| Sharpening | Low teacher temperature, higher student temperature — asymmetry prevents trivial solutions |
| Emergent segmentation | CLS-to-patch attention concentrates on semantically meaningful regions |
| Transfer | DINO features transfer to classification, retrieval, segmentation without fine-tuning |

**Next:** Notebook 07 — CLIP. Contrastive learning across image and text modalities: zero-shot classification, image-text retrieval, and the shared embedding space that powers modern multimodal AI.
